# Harshith: Baseline reproduction on PEMS04 (DSTAGNN, 3 seeds)
**Team project:** TC-STAG (time-conditioned STAD graphs on DSTAGNN) | **Owner:** Harshith | **Dataset:** PEMS04

**Why this notebook matters:** every other result in our paper is compared against *these* numbers. Run it exactly as written; do not change the settings in Cell 1 without telling the group.

**What it produces (in `/kaggle/working/`):**
- `runs/PEMS04_AG_seed{1,2,3}/results.json` (test MAE/RMSE/MAPE, per-horizon, training history)
- `runs/.../test_preds.npz` (test predictions, targets, timestamps; Mihika uses these for the by-regime analysis without needing a GPU)
- `baseline_summary.csv`, `baseline_per_horizon.csv`, `baseline_by_regime.csv`
- `harshith_baseline_outputs.zip` (send this to the group chat)

**Kaggle settings before you start:**
1. Accelerator: **GPU T4 x2** (the notebook uses both GPUs in parallel; P100 also works, it just uses one).
2. Settings > Internet: **On** (needed to `git clone` the authors' repo).
3. Add Data: search Kaggle datasets for **PEMS04** and add one that contains `PEMS04.npz`. The notebook checks the shape is `(16992, 307, 3)`, so a wrong file fails loudly.
4. For the long run use **Save Version > Save & Run All (Commit)** so it keeps running after you close the tab.

## Cell 1: Settings (shared across the team, do not edit)

In [ ]:
import os, sys, json, time, subprocess, shutil, glob, threading, queue
from pathlib import Path

DATASET     = 'PEMS04'
GRAPH       = 'AG'            # 'AG' = learned STAG (the paper's model); 'G' = static road graph (DSTAGNN-G)
SEEDS       = [1, 2, 3]
MAX_EPOCHS  = 60              # team-wide cap (paper used 100-110 without early stopping)
PATIENCE    = 15              # team-wide early stopping on validation loss
SESSION_HOURS_BUDGET = 10.5   # safety: runs stop training after this and still write results (Kaggle kills at 12h)
RUN_TIMING_TEST = True        # 1-epoch timing run first (recommended)

ON_KAGGLE = os.path.exists('/kaggle')
WORK = Path('/kaggle/working') if ON_KAGGLE else Path('./working'); WORK.mkdir(exist_ok=True, parents=True)
REPO = WORK / 'DSTAGNN'
RUNS = WORK / 'runs'; RUNS.mkdir(exist_ok=True)
SESSION_DEADLINE = time.time() + SESSION_HOURS_BUDGET * 3600
print('work dir:', WORK)

## Cell 2: Get the authors' code and check the GPUs

In [ ]:
if not REPO.exists():
    r = subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/SYLan2019/DSTAGNN.git', str(REPO)],
                       capture_output=True, text=True)
    print(r.stdout, r.stderr)
    assert REPO.exists(), 'git clone failed: turn Internet ON in the notebook settings sidebar, then rerun.'
print('repo files:', sorted(os.listdir(REPO)))
print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout or 'NO GPU: set Accelerator to GPU T4 x2')
import torch
N_GPU = torch.cuda.device_count()
print('torch', torch.__version__, '| GPUs visible:', N_GPU)
assert N_GPU >= 1, 'No GPU found. Settings > Accelerator > GPU T4 x2'

## Cell 3: Find `PEMS04.npz` and validate it
The repo ships the PEMS04 graphs (`stag_001_PEMS04.csv`, `strg_001_PEMS04.csv`, `PEMS04.csv`) but **not** the traffic data itself.

In [ ]:
import numpy as np
EXPECT = {'PEMS04': (16992, 307, 3)}
hits = glob.glob('/kaggle/input/**/%s.npz' % DATASET, recursive=True) if ON_KAGGLE else glob.glob('./**/%s.npz' % DATASET, recursive=True)
print('found:', hits)
assert hits, ('Could not find PEMS04.npz. In the right sidebar click Add Data, search "PEMS04", add a dataset that has PEMS04.npz, '
              'then rerun this cell. If none works, upload the file from the STSGCN data pack as your own Kaggle Dataset.')
raw = np.load(hits[0])['data']
print('shape:', raw.shape, '| flow range:', raw[..., 0].min(), raw[..., 0].max())
assert raw.shape == EXPECT[DATASET], 'Wrong file shape %s, expected %s (Table 1 of the paper: 16992 steps, 307 nodes)' % (raw.shape, EXPECT[DATASET])
dst = REPO / 'data' / DATASET / ('%s.npz' % DATASET)
shutil.copy(hits[0], dst); print('copied to', dst)

## Cell 4: Preprocess (builds `PEMS04_r1_d0_w0_dstagnn.npz`: 12-step input windows, 60/20/20 split, z-score from train)

In [ ]:
r = subprocess.run([sys.executable, 'prepareData.py', '--config', 'configurations/%s_dstagnn.conf' % DATASET],
                   cwd=REPO, capture_output=True, text=True)
print(r.stdout[-1500:]); print(r.stderr[-1500:])
prepared = REPO / 'data' / DATASET / ('%s_r1_d0_w0_dstagnn.npz' % DATASET)
assert prepared.exists(), 'prepareData failed, read the error above'
d = np.load(prepared)
print({k: d[k].shape for k in ['train_x', 'val_x', 'test_x', 'test_target', 'test_timestamp']})

## Cell 5: Write the shared training script
`runner.py` is the same file all of us use for every variant (baseline now, our TC-STAG variants later), so the training protocol (seeds, early stopping, metrics, saved outputs) is identical across experiments. Differences from the authors' `train_DSTAGNN_my.py`: the seed is a parameter (theirs is hard-coded to 1), early stopping is added, validation is run after each epoch, and test predictions + timestamps are saved.

In [ ]:
RUNNER = r'''# DSTAGNN runner: one training run = one (dataset, graph, seed). Used by every team notebook.
import os, sys, json, time, argparse, random, configparser, shutil

p = argparse.ArgumentParser()
p.add_argument('--repo', default='.')
p.add_argument('--dataset', default='PEMS04')
p.add_argument('--graph', default='AG', help='AG = learned STAG (paper), G = static road graph')
p.add_argument('--seed', type=int, default=1)
p.add_argument('--gpu', default='0')
p.add_argument('--max_epochs', type=int, default=60)
p.add_argument('--patience', type=int, default=15)
p.add_argument('--out', required=True)
p.add_argument('--deadline', type=float, default=0.0, help='unix time; stop training after this')
args = p.parse_args()

os.environ['CUDA_VISIBLE_DEVICES'] = str(args.gpu)
os.chdir(args.repo)
sys.path.insert(0, os.getcwd())
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
from model.DSTAGNN_my import make_model
from lib.dataloader import load_weighted_adjacency_matrix, load_PA
from lib.utils1 import load_graphdata_channel1, get_adjacency_matrix2
from lib.metrics import masked_mape_np

random.seed(args.seed); np.random.seed(args.seed)
torch.manual_seed(args.seed); torch.cuda.manual_seed_all(args.seed)
torch.backends.cudnn.benchmark = False; torch.backends.cudnn.deterministic = True

cfg = configparser.ConfigParser()
cfg.read('configurations/%s_dstagnn.conf' % args.dataset)
D, T = cfg['Data'], cfg['Training']
N = int(D['num_of_vertices']); n_pred = int(D['num_for_predict']); len_in = int(D['len_input'])
bs = int(T['batch_size']); lr = float(T['learning_rate'])
hours, days, weeks = int(T['num_of_hours']), int(T['num_of_days']), int(T['num_of_weeks'])
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
os.makedirs(args.out, exist_ok=True)

(_, train_loader, _, _, val_loader, _, _, test_loader, test_target, mean, std) = load_graphdata_channel1(
    D['graph_signal_matrix_filename'], hours, days, weeks, DEVICE, bs)
adj_mx = get_adjacency_matrix2(D['adj_filename'], N, id_filename=None)
adj_stag = load_weighted_adjacency_matrix(D['stag_filename'], N)
adj_strg = load_PA(D['strg_filename'])
adj_merge = adj_mx if args.graph == 'G' else adj_stag

net = make_model(DEVICE, int(T['in_channels']), int(T['nb_block']), int(T['in_channels']), int(T['K']),
                 int(T['nb_chev_filter']), int(T['nb_time_filter']), 1, adj_merge, adj_strg, adj_stag,
                 n_pred, len_in, N, int(T['d_model']), int(T['d_k']), int(T['d_k']), int(T['n_heads']))
n_params = int(sum(p_.numel() for p_ in net.parameters()))
print('params:', n_params, 'device:', DEVICE, flush=True)
crit = nn.SmoothL1Loss().to(DEVICE)   # Huber, threshold 1 (as in paper)
opt = optim.Adam(net.parameters(), lr=lr)

def val_loss():
    net.eval(); tot = []
    with torch.no_grad():
        for x, y in val_loader:
            tot.append(crit(net(x), y).item())
    return float(np.mean(tot))

best_path = os.path.join(args.out, 'best.pt')
best_val, best_ep, bad, hist, stopped = float('inf'), -1, 0, [], 'max_epochs'
t_start = time.time()
for ep in range(args.max_epochs):
    t0 = time.time(); net.train(); tl = []
    for x, y in train_loader:
        opt.zero_grad(); loss = crit(net(x), y); loss.backward(); opt.step(); tl.append(loss.detach())
    tr = float(torch.stack(tl).mean()); t_train = time.time() - t0
    vl = val_loss()
    hist.append(dict(epoch=ep, train_loss=tr, val_loss=vl, epoch_sec=time.time() - t0))
    improved = vl < best_val
    if improved:
        best_val, best_ep, bad = vl, ep, 0
        torch.save(net.state_dict(), best_path)
    else:
        bad += 1
    print('ep %d train %.4f val %.4f %s  %.1fs' % (ep, tr, vl, '*' if improved else '', time.time() - t0), flush=True)
    if bad >= args.patience:
        stopped = 'early_stop'; break
    if args.deadline and time.time() > args.deadline:
        stopped = 'deadline'; break

# ---- test at best epoch ----
net.load_state_dict(torch.load(best_path, map_location=DEVICE)); net.eval(); preds = []
with torch.no_grad():
    for x, y in test_loader:
        preds.append(net(x).cpu().numpy())
pred = np.concatenate(preds, 0); tgt = test_target.cpu().numpy()

def m(y, yh):
    return dict(MAE=float(np.mean(np.abs(y - yh))), RMSE=float(np.sqrt(np.mean((y - yh) ** 2))),
                MAPE=float(masked_mape_np(y, yh, 0)))

per_h = [m(tgt[:, :, i], pred[:, :, i]) for i in range(pred.shape[2])]
overall = m(tgt.reshape(-1), pred.reshape(-1))
prep = np.load(D['graph_signal_matrix_filename'].replace('.npz', '') + '_r%d_d%d_w%d_dstagnn.npz' % (hours, days, weeks))
np.savez_compressed(os.path.join(args.out, 'test_preds.npz'), prediction=pred.astype(np.float32),
                    target=tgt.astype(np.float32), timestamp=prep['test_timestamp'])
res = dict(dataset=args.dataset, graph=args.graph, seed=args.seed, best_epoch=best_ep, best_val=best_val,
           epochs_run=len(hist), stopped_by=stopped, max_epochs=args.max_epochs, patience=args.patience,
           mean_epoch_sec=float(np.mean([h['epoch_sec'] for h in hist])), total_sec=time.time() - t_start,
           n_params=n_params, test_overall=overall, test_per_horizon=per_h, history=hist)
json.dump(res, open(os.path.join(args.out, 'results.json'), 'w'), indent=1)
os.remove(best_path)   # keep outputs small; predictions + results are what we need
print('TEST overall:', overall, flush=True)
'''
(WORK / 'runner.py').write_text(RUNNER)
print('wrote', WORK / 'runner.py', len(RUNNER.splitlines()), 'lines')

## Cell 6: 1-epoch timing run (about as long as one epoch)
Tells us the real seconds per epoch on a Kaggle T4. **Copy the printed line into the group chat**, as it decides what the rest of the team can afford.

In [ ]:
def launch(seed, gpu, max_epochs, patience, out, graph=GRAPH):
    cmd = [sys.executable, str(WORK / 'runner.py'), '--repo', str(REPO), '--dataset', DATASET, '--graph', graph,
           '--seed', str(seed), '--gpu', str(gpu), '--max_epochs', str(max_epochs), '--patience', str(patience),
           '--out', str(out), '--deadline', str(SESSION_DEADLINE)]
    log = open(str(out) + '.log', 'w')
    return subprocess.run(cmd, stdout=log, stderr=subprocess.STDOUT)

if RUN_TIMING_TEST:
    out = RUNS / 'timing_test'; out.mkdir(exist_ok=True, parents=True)
    t0 = time.time(); p = launch(seed=0, gpu=0, max_epochs=1, patience=1, out=out)
    print(open(str(out) + '.log').read()[-1500:])
    assert p.returncode == 0, 'Timing run crashed: copy the traceback above and send it to the group.'
    r = json.load(open(out / 'results.json'))
    sec = r['mean_epoch_sec']
    print('\n>>> TIMING: %.0f s/epoch on this GPU | %d params | 1-epoch test MAE %.2f' % (sec, r['n_params'], r['test_overall']['MAE']))
    print('>>> Full run at %d epochs max = up to %.1f h per seed; %d seeds on %d GPU(s) = about %.1f h wall clock' % (
        MAX_EPOCHS, sec * MAX_EPOCHS / 3600, len(SEEDS), N_GPU,
        sec * MAX_EPOCHS / 3600 * -(-len(SEEDS) // N_GPU)))

## Cell 7: Train baseline, seeds 1, 2, 3 (runs in parallel, one seed per GPU)
Logs stream to `runs/PEMS04_AG_seedX.log`. If the session dies, rerun the notebook: finished seeds are skipped.

In [ ]:
jobs = queue.Queue()
for s in SEEDS:
    jobs.put(s)
status = {}

def worker(gpu):
    while True:
        try: s = jobs.get_nowait()
        except queue.Empty: return
        out = RUNS / ('%s_%s_seed%d' % (DATASET, GRAPH, s)); out.mkdir(exist_ok=True, parents=True)
        if (out / 'results.json').exists():
            status[s] = 'skipped (already done)'; continue
        t0 = time.time(); p = launch(s, gpu, MAX_EPOCHS, PATIENCE, out)
        status[s] = 'ok' if p.returncode == 0 else 'FAILED (see %s.log)' % out
        print('seed %d on GPU %d -> %s in %.1f min' % (s, gpu, status[s], (time.time() - t0) / 60), flush=True)

threads = [threading.Thread(target=worker, args=(g,)) for g in range(min(N_GPU, len(SEEDS)))]
[t.start() for t in threads]; [t.join() for t in threads]
print(status)

## Cell 8: Results table vs. the paper (mean ± std over seeds)

In [ ]:
import pandas as pd
rows, hrows = [], []
for f in sorted(glob.glob(str(RUNS / ('%s_%s_seed*/results.json' % (DATASET, GRAPH))))):
    r = json.load(open(f))
    rows.append(dict(seed=r['seed'], MAE=r['test_overall']['MAE'], MAPE=r['test_overall']['MAPE'], RMSE=r['test_overall']['RMSE'],
                     best_epoch=r['best_epoch'], epochs_run=r['epochs_run'], stopped_by=r['stopped_by'],
                     sec_per_epoch=r['mean_epoch_sec'], params=r['n_params']))
    for i, h in enumerate(r['test_per_horizon']):
        hrows.append(dict(seed=r['seed'], horizon=i + 1, **h))
res = pd.DataFrame(rows); print(res.round(3).to_string(index=False))
PAPER = dict(MAE=19.30, MAPE=12.70, RMSE=31.46)   # Table 2, DSTAGNN on PEMS04
summ = pd.DataFrame({'ours_mean': res[['MAE', 'MAPE', 'RMSE']].mean(), 'ours_std': res[['MAE', 'MAPE', 'RMSE']].std(),
                     'paper': pd.Series(PAPER)})
summ['gap_vs_paper_%'] = 100 * (summ['ours_mean'] - summ['paper']) / summ['paper']
print('\n', summ.round(3))
res.to_csv(WORK / 'baseline_summary.csv', index=False)
pd.DataFrame(hrows).to_csv(WORK / 'baseline_per_horizon.csv', index=False)
if (summ['gap_vs_paper_%'].abs() > 3).any():
    print('\nNOTE: more than 3% off the paper. That is OK if the epoch cap (%d) stopped training early. Report OUR retrained numbers as the baseline.' % MAX_EPOCHS)
if (res['stopped_by'] == 'max_epochs').all():
    print('NOTE: all runs hit the epoch cap without early stopping, so the model was probably still improving. Tell the group.')

## Cell 9: Baseline error by traffic regime (this is the table our paper's story depends on)
Regimes: weekday peak (7-10h, 16-19h), weekday off-peak, weekend. **The day-of-week assumption must be verified**: PEMS04 is assumed to start on Monday 2018-01-01 (59 days x 288 steps). The check below prints mean flow per assumed weekday; if Saturday and Sunday are not the two lowest, change `START_DOW` (0 = Monday) until they are.

In [ ]:
START_DOW = 0
daily = raw[:, :, 0].reshape(-1, 288, raw.shape[1]).mean(axis=(1, 2))        # mean flow per day
dow_means = pd.Series(daily).groupby((np.arange(len(daily)) + START_DOW) % 7).mean()
dow_means.index = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
print('mean flow per assumed weekday (weekend should be the lowest two):'); print(dow_means.round(1).to_string())

def regime_of(t):
    tod = t % 288; dow = ((t // 288) + START_DOW) % 7
    peak = ((tod >= 7 * 12) & (tod < 10 * 12)) | ((tod >= 16 * 12) & (tod < 19 * 12))
    return np.where(dow >= 5, 'weekend', np.where(peak, 'weekday_peak', 'weekday_offpeak'))

recs = []
for f in sorted(glob.glob(str(RUNS / ('%s_%s_seed*/test_preds.npz' % (DATASET, GRAPH))))):
    z = np.load(f); reg = regime_of(z['timestamp'][:, 0].astype(int)); seed = int(f.split('seed')[1].split('/')[0])
    for name in ['weekday_peak', 'weekday_offpeak', 'weekend']:
        msk = reg == name
        e = z['prediction'][msk] - z['target'][msk]
        recs.append(dict(seed=seed, regime=name, n_windows=int(msk.sum()), MAE=np.abs(e).mean(), RMSE=np.sqrt((e ** 2).mean())))
reg_df = pd.DataFrame(recs)
print(reg_df.groupby('regime')[['n_windows', 'MAE', 'RMSE']].mean().round(2))
reg_df.to_csv(WORK / 'baseline_by_regime.csv', index=False)

## Cell 10: Package everything to send to the group

In [ ]:
import zipfile
if (REPO / 'data' / DATASET / ('%s.npz' % DATASET)).exists(): os.remove(REPO / 'data' / DATASET / ('%s.npz' % DATASET))
for f in glob.glob(str(RUNS / '*/best.pt')): os.remove(f)
shutil.rmtree(RUNS / 'timing_test', ignore_errors=True)
zip_path = str(WORK / 'harshith_baseline_outputs.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(str(RUNS / '*/*')) + glob.glob(str(RUNS / '*.log')) + glob.glob(str(WORK / 'baseline_*.csv')):
        z.write(f, os.path.relpath(f, WORK))
print('zip:', zip_path, '| %.1f MB' % (os.path.getsize(zip_path) / 1e6))
print('Send to the group: (1) the zip (Output tab of the committed notebook), (2) the TIMING line from Cell 6, (3) the table from Cell 8, (4) the regime table from Cell 9.')